# INGENIERÍA DE VARIABLES

<div style="text-align: justify;">

## 1. Introducción

El presente notebook corresponde a la segunda etapa del proyecto **LiquiditySense**, cuyo objetivo es desarrollar un modelo de Machine Learning orientado a la **predicción del deterioro de la liquidez de las instituciones bancarias**. En esta etapa se realiza la transformación de la base financiera previamente depurada en un conjunto de variables explicativas que permitan representar, desde una perspectiva cuantitativa, la posición de liquidez, la estructura financiera, la calidad de los activos y la evolución temporal de cada institución.

La **ingeniería de variables (Feature Engineering)** constituye una etapa fundamental dentro del desarrollo del modelo, debido a que las variables originales provenientes de los estados financieros no necesariamente contienen, de manera directa, toda la información necesaria para identificar patrones asociados con el deterioro de la liquidez. Por este motivo, a partir de las variables financieras originales se construyen **ratios financieros, indicadores de cobertura, medidas de estructura, tasas de crecimiento interanual y trimestral, así como variables relacionadas con la disponibilidad y evolución de los activos líquidos**.

La construcción de estas variables se realiza respetando la estructura longitudinal de la información. Cada institución bancaria es identificada mediante `id_rssd` y las observaciones son ordenadas cronológicamente mediante la variable `date`. Esto permite calcular correctamente indicadores de variación **trimestre contra trimestre (QoQ)** y **año contra año (YoY)**, evitando mezclar observaciones pertenecientes a diferentes instituciones o periodos.

Un aspecto central de esta etapa es que las variables se construyen utilizando únicamente información disponible hasta cada periodo de observación. Esta consideración es necesaria para preservar la lógica temporal del problema de predicción y evitar la incorporación de **información futura o data leakage**, especialmente debido a que el objetivo final del proyecto consiste en anticipar el deterioro de liquidez del siguiente periodo.

### 1.1 Objetivo del notebook

El objetivo principal de este notebook es transformar la base financiera depurada obtenida en la etapa de preparación de datos en una **base enriquecida de indicadores financieros y variables temporales**, que posteriormente será utilizada en las etapas de análisis exploratorio, definición de la variable objetivo y construcción del modelo predictivo.

De manera específica, esta etapa busca:

* Construir indicadores de **liquidez y cobertura**.
* Representar la **estructura de financiamiento y de activos** de las instituciones.
* Incorporar medidas relacionadas con la **calidad de cartera y solvencia**.
* Calcular tasas de crecimiento **YoY y QoQ** de las principales cuentas financieras.
* Identificar la evolución de los **activos líquidos** a través del tiempo.
* Mantener información sobre determinados patrones de **datos faltantes estructurales** que puedan contener información relevante para el modelo.
* Generar una base de datos consistente y trazable para las siguientes etapas del proyecto.

### 1.2 Metodología

La metodología aplicada parte de la base `base_liquidez.parquet`, generada en el notebook **01_preparacion_base.ipynb**. Sobre esta información se generan variables derivadas mediante operaciones financieras y transformaciones temporales agrupadas por institución bancaria.

Los ratios financieros se calculan principalmente como relaciones entre cuentas del balance, por ejemplo, activos líquidos respecto a activos totales, efectivo respecto a depósitos o créditos respecto a depósitos. Estas transformaciones permiten comparar instituciones de diferente tamaño y obtener medidas más representativas de su estructura financiera.

Para las variables de crecimiento, se utilizan rezagos temporales dentro de cada institución. En particular, las variaciones interanuales consideran un rezago de cuatro observaciones, debido a que la información presenta periodicidad trimestral, mientras que las variaciones trimestrales utilizan el periodo inmediatamente anterior.

Durante la construcción de los indicadores también se consideran situaciones como denominadores iguales a cero, valores infinitos y observaciones con información insuficiente. En estos casos se evita generar valores artificiales y se conserva la información faltante cuando corresponde, permitiendo que posteriormente sea tratada mediante el proceso de preprocesamiento del modelo.

### 1.3 Resultado esperado

Como resultado de esta etapa se obtiene una **base financiera enriquecida**, en la que las variables originales se complementan con indicadores financieros y medidas dinámicas capaces de representar diferentes dimensiones del comportamiento bancario.

Esta base constituye el insumo para las siguientes etapas del proyecto, particularmente para el **análisis exploratorio de datos (EDA)**, la definición del evento de deterioro de liquidez y posteriormente el entrenamiento y evaluación de los modelos de Machine Learning.

En términos generales, el flujo metodológico del proyecto puede representarse como:

**Base financiera original → Preparación y control de calidad → Ingeniería de variables → EDA → Definición del target → Preprocesamiento → Modelado → Evaluación → Registro y seguimiento del modelo con MLflow.**

</div>


## 2. Carga de la base de datos

<div style="text-align: justify;">

En esta primera etapa se carga la base de trabajo generada en el notebook de preparación de datos. La información corresponde a observaciones trimestrales de instituciones bancarias, identificadas mediante `id_rssd` y acompañadas de su respectiva fecha de reporte.

El objetivo de esta etapa es recuperar la información previamente depurada y utilizarla como punto de partida para la construcción de variables financieras, indicadores de liquidez, medidas de crecimiento y posteriormente la variable objetivo del modelo.

La carga se realiza desde la carpeta de datos procesados del proyecto, manteniendo una estructura de rutas relativa al directorio del proyecto. Asimismo, la variable `date` se convierte explícitamente al formato de fecha para garantizar que las operaciones de ordenamiento y cálculo temporal puedan realizarse correctamente.

</div>

**Resultado:** Se obtiene la base `base_liquidez`, que constituye el insumo para la etapa de ingeniería de variables.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================

# RUTA DEL PROYECTO

# ============================================

ROOT_DIR = Path.cwd().parent

ruta_base = (
ROOT_DIR
/ "data"
/ "processed"
/ "base_liquidez.parquet"
)

# ============================================

# CARGAR BASE DE LIQUIDEZ

# ============================================

base_liquidez = pd.read_parquet(
ruta_base,
engine="pyarrow"
)

print("============================================")
print("BASE DE LIQUIDEZ")
print("============================================")
print(f"Registros: {len(base_liquidez):,}")
print(f"Variables: {len(base_liquidez.columns)}")
print(f"Bancos: {base_liquidez['id_rssd'].nunique():,}")
print(f"Fecha mínima: {base_liquidez['date'].min()}")
print(f"Fecha máxima: {base_liquidez['date'].max()}")


BASE DE LIQUIDEZ
Registros: 698,184
Variables: 21
Bancos: 11,339
Fecha mínima: 2000-01-01 00:00:00
Fecha máxima: 2025-10-01 00:00:00


## 2. Revisar la Estructura

## 3. Revisión de la estructura

<div style="text-align: justify;">

Antes de iniciar la construcción de nuevas variables se verifica nuevamente la estructura de la base de datos. Esta revisión permite confirmar que las variables necesarias se encuentran disponibles, que los tipos de datos son adecuados y que la información mantiene la estructura longitudinal definida en la etapa anterior.

La revisión es importante debido a que la ingeniería de variables depende directamente de la correcta identificación de las variables financieras y de la consistencia de la dimensión temporal. En particular, los indicadores de crecimiento requieren que las observaciones de cada banco estén correctamente ordenadas cronológicamente.

</div>

**Resultado:** La base presenta una estructura adecuada para iniciar la construcción de indicadores financieros y variables temporales.

In [3]:
base_liquidez.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 698184 entries, 0 to 698183
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id_rssd           698184 non-null  int32         
 1   date              698184 non-null  datetime64[ns]
 2   assets            698181 non-null  float64       
 3   cash              698181 non-null  float64       
 4   securities        698181 non-null  float64       
 5   deposits          698181 non-null  float64       
 6   demand_deposits   698181 non-null  float64       
 7   time_deposits     698181 non-null  float64       
 8   brokered_dep      698181 non-null  float64       
 9   insured_deposits  695262 non-null  float64       
 10  ln_tot            698181 non-null  float64       
 11  ln_tot_gross      698181 non-null  float64       
 12  ln_re             698181 non-null  float64       
 13  ln_ci             698181 non-null  float64       
 14  ln_c

In [4]:
base_liquidez.head()

,id_rssd,date,assets,cash,securities,deposits,demand_deposits,time_deposits,brokered_dep,insured_deposits,...,ln_tot_gross,ln_re,ln_ci,ln_cons,ln_agr,ln_rre,npl_tot,liab_tot,equity,subdebt
0,37,2000-01-01,64088.0,1436.0,20637.0,52987.0,6446.0,30081.0,0.0,40256.0,...,36620.0,24788.0,790.0,9556.0,37.0,18755.0,360.0,53379.0,10709.0,0.0
1,37,2000-04-01,62982.0,2014.0,20939.0,51772.0,6437.0,29852.0,0.0,40011.0,...,38458.0,25541.0,866.0,10129.0,18.0,19705.0,597.0,52137.0,10845.0,0.0
2,37,2000-07-01,63862.0,1386.0,20395.0,51648.0,6984.0,30208.0,0.0,40156.0,...,41145.0,26315.0,918.0,10753.0,31.0,20677.0,702.0,52773.0,11089.0,0.0
3,37,2000-10-01,62558.0,1465.0,20312.0,51071.0,6461.0,30060.0,0.0,44780.0,...,38235.0,26261.0,1054.0,10458.0,79.0,20767.0,483.0,51354.0,11204.0,0.0
4,37,2001-01-01,65486.0,1729.0,18439.0,53715.0,7024.0,31133.0,0.0,40270.0,...,40507.0,27636.0,711.0,10534.0,106.0,21558.0,426.0,54062.0,11424.0,0.0


## 4. Creación de variables

<div style="text-align: justify;">

En esta etapa se desarrolla la **ingeniería de variables (Feature Engineering)** del proyecto *LiquiditySense*. El objetivo es transformar las variables financieras originales en indicadores que representen diferentes dimensiones de la posición de liquidez, estructura de fondeo, apalancamiento, calidad de cartera y evolución financiera de cada institución.

La utilización de indicadores relativos y tasas de crecimiento permite reducir el efecto de escala entre bancos de diferente tamaño y capturar cambios en la estructura financiera que pueden ser más informativos para la predicción del deterioro de liquidez que los valores absolutos.

Las variables construidas se organizan conceptualmente en tres grandes grupos: indicadores financieros relativos, variables de crecimiento temporal y variables relacionadas con la posición de liquidez. Posteriormente estas variables serán utilizadas como predictores dentro del modelo de Machine Learning.

</div>

**Resultado:** Se amplía la información original mediante la construcción de variables financieras y temporales orientadas a explicar el comportamiento futuro de la liquidez bancaria.

In [5]:
# ============================================
# REVISIÓN DE DEPÓSITOS
# ============================================

print("Depósitos iguales a cero:")
print((base_liquidez["deposits"] == 0).sum())

print("\nDepósitos nulos:")
print(base_liquidez["deposits"].isna().sum())

print("\nEstadísticos de depósitos:")
print(base_liquidez["deposits"].describe())

Depósitos iguales a cero:
6904

Depósitos nulos:
3

Estadísticos de depósitos:
count    6.981810e+05
mean     1.660488e+06
std      3.107008e+07
min      0.000000e+00
25%      5.682600e+04
50%      1.267060e+05
75%      3.108400e+05
max      2.697842e+09
Name: deposits, dtype: float64


### 4.1. Índice de Liquidez Inmediata

<div style="text-align: justify;">

Se construye un indicador de liquidez inmediata relacionando los activos líquidos de mayor disponibilidad, representados por el efectivo (`cash`) y los valores negociables (`securities`), con el total de depósitos (`deposits`).

Este indicador busca aproximar la capacidad de una institución financiera para hacer frente a sus obligaciones asociadas a depósitos utilizando recursos líquidos disponibles. Al expresarlo como una proporción de los depósitos, se obtiene una medida comparable entre instituciones de diferente tamaño.

El tratamiento de depósitos iguales a cero es necesario para evitar divisiones indefinidas. En estos casos el denominador se transforma temporalmente en valor nulo, evitando la generación de valores infinitos.

</div>

**Resultado:** Se genera la variable `liquidez_inmediata` y se revisan sus estadísticos descriptivos y valores extremos.

In [6]:
# ============================================
# LIQUIDEZ INMEDIATA
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["liquidez_inmediata"] = (
    base_liquidez["cash"] + base_liquidez["securities"]
) / denominador

In [7]:
# ============================================
# VALIDACIÓN DEL INDICADOR
# ============================================

print("============================================")
print("LIQUIDEZ INMEDIATA")
print("============================================")

print(
    base_liquidez["liquidez_inmediata"].describe()
)

LIQUIDEZ INMEDIATA
count    691277.000000
mean          2.058567
std         302.241646
min           0.000000
25%           0.214402
50%           0.325396
75%           0.469879
max      165143.000000
Name: liquidez_inmediata, dtype: float64


In [8]:
# ============================================
# ANÁLISIS DE VALORES EXTREMOS
# ============================================

percentiles_liquidez = (
    base_liquidez["liquidez_inmediata"]
    .quantile([0.90, 0.95, 0.99, 0.995, 0.999, 1.00])
)

print(percentiles_liquidez)

0.900         0.633442
0.950         0.748471
0.990         1.148314
0.995         4.380481
0.999       108.482634
1.000    165143.000000
Name: liquidez_inmediata, dtype: float64


In [9]:
# ============================================
# MAYORES VALORES DE LIQUIDEZ INMEDIATA
# ============================================

extremos = (
    base_liquidez[
        [
            "id_rssd",
            "date",
            "cash",
            "securities",
            "deposits",
            "assets",
            "liquidez_inmediata"
        ]
    ]
    .sort_values(
        "liquidez_inmediata",
        ascending=False
    )
    .head(20)
)

print(extremos)

        id_rssd       date       cash  securities  deposits     assets  \
563965  1444021 2009-01-01    28544.0    466885.0       3.0  1484057.0   
556743  1216826 2024-01-01   247486.0         0.0       2.0   257237.0   
563944  1444021 2003-10-01    25632.0    237908.0       3.0   712103.0   
563945  1444021 2004-01-01    26843.0    164866.0       3.0   682976.0   
697813  5686089 2023-07-01    92435.0         0.0       2.0    99801.0   
556742  1216826 2023-10-01   258615.0         0.0       6.0   269244.0   
402350   750864 2017-01-01   113849.0         0.0       3.0   126367.0   
563943  1444021 2003-07-01    24369.0    202268.0       9.0   676269.0   
67621    128511 2016-01-01    97119.0         0.0       4.0    97218.0   
87522    167565 2000-10-01   134460.0         0.0      10.0   278752.0   
87525    167565 2001-07-01   121431.0         0.0      10.0   258751.0   
87521    167565 2000-07-01   119718.0         0.0      10.0   267723.0   
87524    167565 2001-04-01   117018.0 

In [10]:
# ============================================
# REGISTROS CON DEPÓSITOS MUY BAJOS
# ============================================

casos_depositos_bajos = (
    base_liquidez[
        base_liquidez["deposits"].between(1, 1000)
    ][
        [
            "id_rssd",
            "date",
            "cash",
            "securities",
            "deposits",
            "assets",
            "liquidez_inmediata"
        ]
    ]
    .sort_values(
        "liquidez_inmediata",
        ascending=False
    )
    .head(20)
)

print(casos_depositos_bajos)

        id_rssd       date       cash  securities  deposits     assets  \
563965  1444021 2009-01-01    28544.0    466885.0       3.0  1484057.0   
556743  1216826 2024-01-01   247486.0         0.0       2.0   257237.0   
563944  1444021 2003-10-01    25632.0    237908.0       3.0   712103.0   
563945  1444021 2004-01-01    26843.0    164866.0       3.0   682976.0   
697813  5686089 2023-07-01    92435.0         0.0       2.0    99801.0   
556742  1216826 2023-10-01   258615.0         0.0       6.0   269244.0   
402350   750864 2017-01-01   113849.0         0.0       3.0   126367.0   
563943  1444021 2003-07-01    24369.0    202268.0       9.0   676269.0   
67621    128511 2016-01-01    97119.0         0.0       4.0    97218.0   
87522    167565 2000-10-01   134460.0         0.0      10.0   278752.0   
87525    167565 2001-07-01   121431.0         0.0      10.0   258751.0   
87521    167565 2000-07-01   119718.0         0.0      10.0   267723.0   
87524    167565 2001-04-01   117018.0 

#### 4.1.1. Análisis de la variable Liquidez

<div style="text-align: justify;">

Una vez construido el indicador de liquidez inmediata, se realiza un análisis de su distribución y de sus valores extremos. Esta revisión permite determinar si existen observaciones con niveles excepcionalmente elevados que puedan estar asociados a denominadores muy pequeños o a características particulares de determinadas instituciones.

Se utilizan percentiles altos de la distribución y diferentes puntos de corte para cuantificar la magnitud y frecuencia de los valores extremos. Adicionalmente, se analizan los registros con niveles muy bajos de depósitos, debido a que pequeñas variaciones en el denominador pueden generar ratios elevados aun cuando los valores absolutos de liquidez no sean extraordinarios.

Este análisis no implica eliminar automáticamente los valores extremos. Su finalidad es comprender su origen y determinar posteriormente el tratamiento más apropiado dentro del proceso de modelado.

</div>

**Resultado:** Se identifican y caracterizan los valores extremos de `liquidez_inmediata`, manteniendo la información para su posterior tratamiento dentro del pipeline de Machine Learning.

In [11]:
# ============================================
# MAGNITUD DE VALORES EXTREMOS
# ============================================

limites = [1, 2, 5, 10, 50, 100, 1000]

for limite in limites:
    cantidad = (
        base_liquidez["liquidez_inmediata"] > limite
    ).sum()

    porcentaje = (
        cantidad / len(base_liquidez) * 100
    )

    print(
        f"> {limite:>4}: "
        f"{cantidad:>8,} registros "
        f"({porcentaje:.2f}%)"
    )

>    1:   10,298 registros (1.47%)
>    2:    4,331 registros (0.62%)
>    5:    3,245 registros (0.46%)
>   10:    2,278 registros (0.33%)
>   50:    1,103 registros (0.16%)
>  100:      756 registros (0.11%)
> 1000:      122 registros (0.02%)


In [12]:
# ============================================
# PERCENTILES DE LIQUIDEZ INMEDIATA
# ============================================

print(
    base_liquidez["liquidez_inmediata"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500      0.325396
0.750      0.469879
0.900      0.633442
0.950      0.748471
0.990      1.148314
0.995      4.380481
0.999    108.482634
Name: liquidez_inmediata, dtype: float64


In [13]:
# ============================================
# DEPÓSITOS MUY BAJOS
# ============================================

for limite in [10, 100, 1000, 10000]:
    
    cantidad = (
        (base_liquidez["deposits"] > 0) &
        (base_liquidez["deposits"] <= limite)
    ).sum()

    porcentaje = cantidad / len(base_liquidez) * 100

    print(
        f"Depósitos 1-{limite:,}: "
        f"{cantidad:,} registros "
        f"({porcentaje:.2f}%)"
    )

Depósitos 1-10: 53 registros (0.01%)
Depósitos 1-100: 180 registros (0.03%)
Depósitos 1-1,000: 3,177 registros (0.46%)
Depósitos 1-10,000: 11,002 registros (1.58%)


### 4.2. Cobertura de efectivo sobre depósitos

<div style="text-align: justify;">

Se construye el indicador `cash_deposits`, definido como la relación entre el efectivo disponible (`cash`) y los depósitos (`deposits`).

Esta variable permite medir qué proporción de los depósitos se encuentra respaldada directamente por efectivo. Desde una perspectiva de liquidez, una mayor cobertura de efectivo puede representar una mayor capacidad inmediata para atender retiros o requerimientos de fondos.

El indicador se calcula utilizando los depósitos como denominador y se controla previamente la existencia de valores iguales a cero para evitar divisiones indefinidas y la generación de valores infinitos.

</div>

**Resultado:** Se genera `cash_deposits` y se validan sus valores nulos, infinitos y principales percentiles.

In [14]:
# ============================================
# CASH / DEPÓSITOS
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["cash_deposits"] = (
    base_liquidez["cash"] / denominador
)

print("============================================")
print("CASH / DEPÓSITOS")
print("============================================")

print(
    base_liquidez["cash_deposits"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(base_liquidez["cash_deposits"]).sum()
)

print(
    "Valores nulos:",
    base_liquidez["cash_deposits"].isna().sum()
)

CASH / DEPÓSITOS
count    691277.000000
mean          0.989604
std         182.439908
min          -0.003067
25%           0.034755
50%           0.057941
75%           0.111818
max      123743.000000
Name: cash_deposits, dtype: float64

Valores infinitos: 0
Valores nulos: 6907


In [15]:
# ============================================
# PERCENTILES
# ============================================

print(
    base_liquidez["cash_deposits"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500     0.057941
0.750     0.111818
0.900     0.203493
0.950     0.285490
0.990     0.600000
0.995     1.093048
0.999    46.386608
Name: cash_deposits, dtype: float64


### 4.3. Cobertura de valores sobre depósitos

<div style="text-align: justify;">

Se construye `securities_deposits`, que relaciona los valores negociables mantenidos por la institución con el total de depósitos.

El indicador permite complementar la medición de liquidez inmediata, considerando que los valores negociables pueden constituir una fuente adicional de activos líquidos o relativamente líquidos para atender obligaciones financieras.

Su utilización permite capturar una dimensión diferente a la cobertura exclusivamente basada en efectivo y contribuye a representar la composición de los activos líquidos de cada banco.

</div>

**Resultado:** Se genera `securities_deposits` y se verifica su distribución, valores extremos, nulos e infinitos.

In [16]:
# ============================================
# SECURITIES / DEPÓSITOS
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["securities_deposits"] = (
    base_liquidez["securities"] / denominador
)

print("============================================")
print("SECURITIES / DEPÓSITOS")
print("============================================")

print(
    base_liquidez["securities_deposits"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(base_liquidez["securities_deposits"]).sum()
)

print(
    "Valores nulos:",
    base_liquidez["securities_deposits"].isna().sum()
)

SECURITIES / DEPÓSITOS
count    691277.000000
mean          1.068964
std         223.742900
min           0.000000
25%           0.126471
50%           0.234999
75%           0.372504
max      155628.333333
Name: securities_deposits, dtype: float64

Valores infinitos: 0
Valores nulos: 6907


In [17]:
print(
    base_liquidez["securities_deposits"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500     0.234999
0.750     0.372504
0.900     0.526061
0.950     0.634743
0.990     0.930199
0.995     1.292808
0.999    29.203168
Name: securities_deposits, dtype: float64


### 4.4. Índice de créditos sobre depósitos

<div style="text-align: justify;">

Se construye el indicador `loan_deposit`, equivalente al **Loan-to-Deposit Ratio (LTD)**, mediante la relación entre la cartera de créditos (`ln_tot`) y los depósitos (`deposits`).

Este indicador permite evaluar qué proporción de los depósitos se encuentra destinada al financiamiento de la cartera crediticia. Un nivel elevado puede reflejar una mayor utilización de los recursos captados para otorgar créditos y, por tanto, una menor disponibilidad relativa de fondos para atender necesidades inmediatas de liquidez.

La variable incorpora una dimensión de transformación de fondeo en activos crediticios, complementando los indicadores directamente relacionados con los activos líquidos.

</div>

**Resultado:** Se genera `loan_deposit` y se validan sus valores extremos y posibles problemas derivados del denominador.

In [18]:
# ============================================
# ÍNDICE DE CRÉDITOS SOBRE DEPÓSITOS
# Loan-to-Deposit Ratio (LTD)
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["loan_deposit"] = (
    base_liquidez["ln_tot"] / denominador
)

print("============================================")
print("ÍNDICE DE CRÉDITOS SOBRE DEPÓSITOS (LTD)")
print("============================================")

print(
    base_liquidez["loan_deposit"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(base_liquidez["loan_deposit"]).sum()
)

print(
    "Valores nulos:",
    base_liquidez["loan_deposit"].isna().sum()
)

ÍNDICE DE CRÉDITOS SOBRE DEPÓSITOS (LTD)
count    691277.000000
mean          5.724101
std         453.049861
min           0.000000
25%           0.634764
50%           0.782252
75%           0.909341
max      172875.500000
Name: loan_deposit, dtype: float64

Valores infinitos: 0
Valores nulos: 6907


In [19]:
print(
    base_liquidez["loan_deposit"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500     0.782252
0.750     0.909341
0.900     1.015542
0.950     1.088590
0.990     1.318604
0.995     1.556840
0.999    28.437776
Name: loan_deposit, dtype: float64


### 4.5. Índice de créditos sobre activos

<div style="text-align: justify;">

Se construye `loan_assets`, que representa la participación de la cartera de créditos sobre el total de activos.

Este indicador permite identificar la estructura de utilización de los activos de cada institución. Una mayor proporción de activos concentrada en cartera crediticia puede estar asociada con una menor proporción relativa de recursos disponibles en instrumentos líquidos.

La variable permite, por tanto, incorporar al modelo información sobre la estructura del balance y el grado de orientación de los activos hacia la actividad crediticia.

</div>

**Resultado:** Se genera `loan_assets` y se revisan sus principales estadísticos y valores extremos.

In [20]:
# ============================================
# ÍNDICE DE CRÉDITOS SOBRE ACTIVOS
# ============================================

denominador = base_liquidez["assets"].replace(0, np.nan)

base_liquidez["loan_assets"] = (
    base_liquidez["ln_tot"] / denominador
)

print("============================================")
print("ÍNDICE DE CRÉDITOS SOBRE ACTIVOS")
print("============================================")

print(
    base_liquidez["loan_assets"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(base_liquidez["loan_assets"]).sum()
)

print(
    "Valores nulos:",
    base_liquidez["loan_assets"].isna().sum()
)

ÍNDICE DE CRÉDITOS SOBRE ACTIVOS
count    698178.000000
mean          0.623899
std           0.178966
min           0.000000
25%           0.531821
50%           0.654678
75%           0.750991
max           1.207267
Name: loan_assets, dtype: float64

Valores infinitos: 0
Valores nulos: 6


In [21]:
print(
    base_liquidez["loan_assets"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500    0.654678
0.750    0.750991
0.900    0.819588
0.950    0.853759
0.990    0.911399
0.995    0.930478
0.999    0.966290
Name: loan_assets, dtype: float64


### 4.6. Depósitos sobre activos

<div style="text-align: justify;">

Se construye `deposits_assets`, definido como la proporción de depósitos respecto al total de activos.

Esta variable permite aproximar el peso del fondeo mediante depósitos dentro de la estructura financiera de la institución. La relación entre depósitos y activos resulta relevante para caracterizar la estructura de financiación del balance y su posible exposición a cambios en el comportamiento de los depositantes.

</div>

**Resultado:** Se genera `deposits_assets` y se valida su distribución y consistencia.

In [22]:
# ============================================
# DEPÓSITOS SOBRE ACTIVOS
# ============================================

denominador = base_liquidez["assets"].replace(0, np.nan)

base_liquidez["deposits_assets"] = (
    base_liquidez["deposits"] / denominador
)

print("============================================")
print("ÍNDICE DE DEPÓSITOS SOBRE ACTIVOS")
print("============================================")

print(
    base_liquidez["deposits_assets"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(base_liquidez["deposits_assets"]).sum()
)

print(
    "Valores nulos:",
    base_liquidez["deposits_assets"].isna().sum()
)

ÍNDICE DE DEPÓSITOS SOBRE ACTIVOS
count    698178.000000
mean          0.818676
std           0.130292
min           0.000000
25%           0.799067
50%           0.848819
75%           0.883344
max           1.151905
Name: deposits_assets, dtype: float64

Valores infinitos: 0
Valores nulos: 6


In [23]:
print(
    base_liquidez["deposits_assets"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500    0.848819
0.750    0.883344
0.900    0.904333
0.950    0.914133
0.990    0.933503
0.995    0.944044
0.999    0.972087
Name: deposits_assets, dtype: float64


### 4.7. Dependencia de depósitos broker

<div style="text-align: justify;">

Se construye `brokered_deposits_ratio`, que mide la participación de los depósitos broker dentro del total de depósitos.

Este indicador busca capturar la estructura y composición del fondeo bancario. La dependencia relativa de depósitos broker puede representar una característica particular de la estrategia de captación de recursos de una institución y, por tanto, aportar información adicional para explicar posibles episodios de deterioro de liquidez.

</div>

**Resultado:** Se genera `brokered_deposits_ratio` y se revisa su distribución y comportamiento extremo.

In [24]:
# ============================================
# DEPENDENCIA DE DEPÓSITOS BROKER
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["brokered_deposits_ratio"] = (
    base_liquidez["brokered_dep"] / denominador
)

print("============================================")
print("DEPENDENCIA DE DEPÓSITOS BROKER")
print("============================================")

print(
    base_liquidez["brokered_deposits_ratio"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["brokered_deposits_ratio"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["brokered_deposits_ratio"].isna().sum()
)

DEPENDENCIA DE DEPÓSITOS BROKER
count    691277.000000
mean          0.031158
std           0.144466
min           0.000000
25%           0.000000
50%           0.000000
75%           0.020408
max          70.020559
Name: brokered_deposits_ratio, dtype: float64

Valores infinitos: 0
Valores nulos: 6907


In [25]:
print(
    base_liquidez["brokered_deposits_ratio"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500    0.000000
0.750    0.020408
0.900    0.095560
0.950    0.165052
0.990    0.405025
0.995    0.583199
0.999    0.990746
Name: brokered_deposits_ratio, dtype: float64


### 4.8. Dependencia de depósitos asegurados

<div style="text-align: justify;">

Se construye `insured_deposits_ratio`, relacionando los depósitos asegurados con el total de depósitos.

La variable permite incorporar información sobre la composición del fondeo y las características de los depósitos mantenidos por cada institución. Debido a que durante la preparación de la base se identificaron observaciones con información faltante en `insured_deposits`, se conserva dicha información como parte del proceso de ingeniería de variables para que posteriormente pueda ser tratada mediante el esquema de imputación definido para el modelo.

</div>

**Resultado:** Se genera `insured_deposits_ratio` y se verifican sus valores nulos e infinitos.

In [26]:
# ============================================
# DEPENDENCIA / COBERTURA DE DEPÓSITOS ASEGURADOS
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["insured_deposits_ratio"] = (
    base_liquidez["insured_deposits"] / denominador
)

print("============================================")
print("DEPÓSITOS ASEGURADOS SOBRE DEPÓSITOS")
print("============================================")

print(
    base_liquidez["insured_deposits_ratio"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["insured_deposits_ratio"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["insured_deposits_ratio"].isna().sum()
)

DEPÓSITOS ASEGURADOS SOBRE DEPÓSITOS
count    691255.000000
mean          0.662009
std           0.168506
min           0.000000
25%           0.576146
50%           0.686992
75%           0.777651
max           1.151188
Name: insured_deposits_ratio, dtype: float64

Valores infinitos: 0
Valores nulos: 6929


In [27]:
print(
    base_liquidez["insured_deposits_ratio"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

0.500    0.686992
0.750    0.777651
0.900    0.849422
0.950    0.888044
0.990    0.953947
0.995    0.981209
0.999    1.001383
Name: insured_deposits_ratio, dtype: float64


### 4.9. Estructura de depósitos a plazo

<div style="text-align: justify;">

Se construye `time_deposits_ratio`, que representa la proporción de depósitos a plazo respecto al total de depósitos.

Este indicador permite caracterizar la estructura temporal del fondeo. Los depósitos a plazo presentan características diferentes respecto a depósitos de disponibilidad inmediata, por lo que su participación relativa puede aportar información sobre la estabilidad y composición de las fuentes de financiación.

</div>

**Resultado:** Se genera `time_deposits_ratio` y se revisan sus estadísticos y percentiles.

In [28]:
# ============================================
# DEPÓSITOS A PLAZO SOBRE DEPÓSITOS
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["time_deposits_ratio"] = (
    base_liquidez["time_deposits"] / denominador
)

print("============================================")
print("DEPÓSITOS A PLAZO SOBRE DEPÓSITOS")
print("============================================")

print(
    base_liquidez["time_deposits_ratio"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["time_deposits_ratio"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["time_deposits_ratio"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["time_deposits_ratio"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

DEPÓSITOS A PLAZO SOBRE DEPÓSITOS
count    691277.000000
mean          0.401877
std           0.183387
min           0.000000
25%           0.268064
50%           0.402652
75%           0.528123
max           1.001988
Name: time_deposits_ratio, dtype: float64

Valores infinitos: 0
Valores nulos: 6907

Percentiles:
0.500    0.402652
0.750    0.528123
0.900    0.630559
0.950    0.695148
0.990    0.874969
0.995    0.987040
0.999    1.000000
Name: time_deposits_ratio, dtype: float64


### 4.10. Concentración de depósitos

<div style="text-align: justify;">

Se construye `demand_deposits_ratio`, que mide la proporción de depósitos a la vista respecto al total de depósitos.

Esta variable permite complementar el análisis de la estructura de fondeo, identificando qué parte de los recursos captados corresponde a depósitos potencialmente más sensibles a movimientos de corto plazo.

Su incorporación permite al modelo disponer de información sobre la composición del pasivo y no únicamente sobre el volumen total de depósitos.

</div>

**Resultado:** Se genera `demand_deposits_ratio` y se valida su distribución y comportamiento extremo.

In [29]:

# ============================================
# DEPÓSITOS A LA VISTA SOBRE DEPÓSITOS
# ============================================

denominador = base_liquidez["deposits"].replace(0, np.nan)

base_liquidez["demand_deposits_ratio"] = (
    base_liquidez["demand_deposits"] / denominador
)

print("============================================")
print("DEPÓSITOS A LA VISTA SOBRE DEPÓSITOS")
print("============================================")

print(
    base_liquidez["demand_deposits_ratio"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["demand_deposits_ratio"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["demand_deposits_ratio"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["demand_deposits_ratio"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

DEPÓSITOS A LA VISTA SOBRE DEPÓSITOS
count    691277.000000
mean          0.169907
std           0.120637
min           0.000000
25%           0.090213
50%           0.147052
75%           0.221313
max           1.000207
Name: demand_deposits_ratio, dtype: float64

Valores infinitos: 0
Valores nulos: 6907

Percentiles:
0.500    0.147052
0.750    0.221313
0.900    0.312447
0.950    0.383254
0.990    0.576611
0.995    0.726689
0.999    1.000000
Name: demand_deposits_ratio, dtype: float64


### 4.11. Apalancamiento financiero

<div style="text-align: justify;">

Se construye `leverage_ratio` mediante la relación entre los pasivos totales (`liab_tot`) y los activos (`assets`).

Este indicador permite aproximar el grado en que los activos de la institución se encuentran financiados mediante obligaciones con terceros. La estructura de apalancamiento puede influir sobre la capacidad de una institución para absorber shocks financieros y mantener niveles adecuados de liquidez.

La variable incorpora, por tanto, una dimensión de solvencia y estructura financiera complementaria a los indicadores específicos de liquidez.

</div>

**Resultado:** Se genera `leverage_ratio` y se revisan sus estadísticos y valores extremos.

In [30]:
# ============================================
# APALANCAMIENTO FINANCIERO
# ============================================

denominador = base_liquidez["assets"].replace(0, np.nan)

base_liquidez["leverage_ratio"] = (
    base_liquidez["liab_tot"] / denominador
)

print("============================================")
print("APALANCAMIENTO FINANCIERO")
print("============================================")

print(
    base_liquidez["leverage_ratio"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["leverage_ratio"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["leverage_ratio"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["leverage_ratio"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

APALANCAMIENTO FINANCIERO
count    698178.000000
mean          0.877533
std           0.103109
min           0.000000
25%           0.875543
50%           0.898065
75%           0.913356
max           6.194690
Name: leverage_ratio, dtype: float64

Valores infinitos: 0
Valores nulos: 6

Percentiles:
0.500    0.898065
0.750    0.913356
0.900    0.924487
0.950    0.931269
0.990    0.951840
0.995    0.964855
0.999    0.989807
Name: leverage_ratio, dtype: float64


### 4.12. Cobertura de pasivos con activos líquidos

<div style="text-align: justify;">

Se construye `liquid_assets_liabilities`, relacionando los activos líquidos, definidos como efectivo más valores negociables, con los pasivos totales.

Este indicador permite evaluar en qué medida los recursos líquidos disponibles podrían cubrir las obligaciones totales de la institución. Constituye una medida complementaria de liquidez estructural, debido a que relaciona directamente los activos líquidos con la totalidad de los pasivos.

</div>

**Resultado:** Se genera `liquid_assets_liabilities` y se validan sus valores nulos, infinitos y extremos.

In [31]:
# ============================================
# COBERTURA DE PASIVOS CON ACTIVOS LÍQUIDOS
# ============================================

denominador = base_liquidez["liab_tot"].replace(0, np.nan)

base_liquidez["liquid_assets_liabilities"] = (
    (
        base_liquidez["cash"]
        + base_liquidez["securities"]
    )
    / denominador
)

print("============================================")
print("COBERTURA DE PASIVOS CON ACTIVOS LÍQUIDOS")
print("============================================")

print(
    base_liquidez["liquid_assets_liabilities"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["liquid_assets_liabilities"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["liquid_assets_liabilities"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["liquid_assets_liabilities"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999
        ]
    )
)

COBERTURA DE PASIVOS CON ACTIVOS LÍQUIDOS
count    698014.000000
mean          0.943198
std          45.286620
min           0.000000
25%           0.199409
50%           0.305391
75%           0.446461
max       13381.000000
Name: liquid_assets_liabilities, dtype: float64

Valores infinitos: 0
Valores nulos: 170

Percentiles:
0.500     0.305391
0.750     0.446461
0.900     0.608116
0.950     0.728765
0.990     2.467991
0.995     8.494489
0.999    59.762160
Name: liquid_assets_liabilities, dtype: float64


### 4.13. Ratio de capital sobre activos

<div style="text-align: justify;">

Se construye `capital_assets_ratio`, definido como la relación entre el patrimonio (`equity`) y los activos totales.

Este indicador incorpora información sobre la estructura de capital de la institución y su capacidad relativa de absorción de pérdidas. Aunque el objetivo principal del proyecto es la predicción del deterioro de liquidez, la posición patrimonial puede estar relacionada con la capacidad de una institución para enfrentar escenarios adversos.

Por esta razón, se incorpora como variable explicativa adicional dentro del conjunto de características financieras.

</div>

**Resultado:** Se genera `capital_assets_ratio` y se revisa su distribución y posibles valores extremos.

In [32]:
# ============================================
# CAPITAL SOBRE ACTIVOS
# ============================================

denominador = base_liquidez["assets"].replace(0, np.nan)

base_liquidez["capital_assets_ratio"] = (
    base_liquidez["equity"] / denominador
)

print("============================================")
print("CAPITAL SOBRE ACTIVOS")
print("============================================")

print(
    base_liquidez["capital_assets_ratio"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["capital_assets_ratio"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["capital_assets_ratio"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["capital_assets_ratio"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CAPITAL SOBRE ACTIVOS
count    698178.000000
mean          0.122409
std           0.103117
min          -5.194690
25%           0.086594
50%           0.101883
75%           0.124400
max           1.000495
Name: capital_assets_ratio, dtype: float64

Valores infinitos: 0
Valores nulos: 6

Percentiles:
0.010    0.047999
0.050    0.068645
0.500    0.101883
0.750    0.124400
0.900    0.160042
0.950    0.201449
0.990    0.820063
0.999    0.991195
Name: capital_assets_ratio, dtype: float64


### 4.14. Calidad de cartera crediticia

<div style="text-align: justify;">

Se construye `npl_ratio`, definido como la relación entre la cartera vencida o problemática (`npl_tot`) y la cartera total (`ln_tot`).

Este indicador incorpora una dimensión de calidad crediticia al modelo. Un deterioro en la calidad de cartera puede afectar los flujos financieros de una institución, incrementar las necesidades de provisiones y reducir su capacidad de generar recursos líquidos.

La inclusión de esta variable permite que el modelo considere no solamente la posición actual de liquidez, sino también la calidad de los activos crediticios que forman parte del balance.

</div>

**Resultado:** Se genera `npl_ratio` y se analizan sus valores extremos, nulos e infinitos.

In [33]:
# ============================================
# RATIO DE CARTERA VENCIDA / NPL
# ============================================

denominador = base_liquidez["ln_tot"].replace(0, np.nan)

base_liquidez["npl_ratio"] = (
    base_liquidez["npl_tot"] / denominador
)

print("============================================")
print("RATIO DE CARTERA VENCIDA / NPL")
print("============================================")

print(
    base_liquidez["npl_ratio"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["npl_ratio"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["npl_ratio"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["npl_ratio"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

RATIO DE CARTERA VENCIDA / NPL
count    687068.000000
mean          0.013797
std           0.024315
min           0.000000
25%           0.001835
50%           0.006661
75%           0.016370
max           1.000000
Name: npl_ratio, dtype: float64

Valores infinitos: 0
Valores nulos: 11116

Percentiles:
0.010    0.000000
0.050    0.000000
0.500    0.006661
0.750    0.016370
0.900    0.033321
0.950    0.050176
0.990    0.109615
0.999    0.240656
Name: npl_ratio, dtype: float64


### 4.15. Crecimiento interanual de la cartera

<div style="text-align: justify;">

Se construye `loan_growth_yoy`, que mide la variación porcentual de la cartera de créditos respecto al mismo período del año anterior.

Dado que la información presenta una frecuencia trimestral, se utiliza un rezago de cuatro períodos para realizar la comparación interanual. El cálculo se efectúa individualmente para cada institución mediante la agrupación por `id_rssd`, evitando comparar información de diferentes bancos.

Esta variable permite capturar cambios en el ritmo de expansión o contracción de la cartera, incorporando una dimensión dinámica que no está disponible en los valores absolutos del balance.

</div>

**Resultado:** Se genera `loan_growth_yoy`, se identifican valores nulos asociados principalmente a la ausencia de períodos históricos suficientes y se corrigen posibles valores infinitos.

In [34]:
# ============================================
# CRECIMIENTO INTERANUAL DE LA CARTERA
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
).copy()

base_liquidez["loan_growth_yoy"] = (
    base_liquidez
    .groupby("id_rssd")["ln_tot"]
    .pct_change(periods=4)
)

print("============================================")
print("CRECIMIENTO INTERANUAL DE LA CARTERA")
print("============================================")

print(
    base_liquidez["loan_growth_yoy"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["loan_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["loan_growth_yoy"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["loan_growth_yoy"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO INTERANUAL DE LA CARTERA
count    6.439750e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -6.379724e-03
50%      5.547102e-02
75%      1.303422e-01
max               inf
Name: loan_growth_yoy, dtype: float64

Valores infinitos: 306
Valores nulos: 54209

Percentiles:
0.010    -0.232972
0.050    -0.113946
0.500     0.055471
0.750     0.130342
0.900     0.250708
0.950     0.402449
0.990     1.439825
0.999    31.749448
Name: loan_growth_yoy, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\3294751032.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=4)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [35]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["loan_growth_yoy"] = (
    base_liquidez["loan_growth_yoy"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(base_liquidez["loan_growth_yoy"]).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["loan_growth_yoy"].isna().sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 54515


### 4.16. Crecimiento interanual de depósitos

<div style="text-align: justify;">

Se construye `deposit_growth_yoy`, que mide la variación porcentual de los depósitos respecto al mismo trimestre del año anterior.

El crecimiento de los depósitos constituye una variable relevante para el análisis de liquidez debido a que representa la evolución de una de las principales fuentes de fondeo de las instituciones financieras.

La comparación interanual permite reducir el efecto de la estacionalidad trimestral y facilita la identificación de cambios sostenidos en la dinámica de captaciones.

</div>

**Resultado:** Se genera `deposit_growth_yoy`, se validan sus valores nulos e infinitos y se documenta su distribución.

In [36]:
# ============================================
# CRECIMIENTO INTERANUAL DE DEPÓSITOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
).copy()

base_liquidez["deposit_growth_yoy"] = (
    base_liquidez
    .groupby("id_rssd")["deposits"]
    .pct_change(periods=4)
)

print("============================================")
print("CRECIMIENTO INTERANUAL DE DEPÓSITOS")
print("============================================")

print(
    base_liquidez["deposit_growth_yoy"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["deposit_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["deposit_growth_yoy"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["deposit_growth_yoy"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO INTERANUAL DE DEPÓSITOS
count    6.475860e+05
mean              inf
std               NaN
min     -1.000000e+00
25%      0.000000e+00
50%      4.932079e-02
75%      1.188284e-01
max               inf
Name: deposit_growth_yoy, dtype: float64

Valores infinitos: 104
Valores nulos: 50598

Percentiles:
0.010    -0.185473
0.050    -0.080109
0.500     0.049321
0.750     0.118828
0.900     0.235728
0.950     0.381263
0.990     1.286830
0.999    14.054493
Name: deposit_growth_yoy, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\357665582.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=4)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [37]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["deposit_growth_yoy"] = (
    base_liquidez["deposit_growth_yoy"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["deposit_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["deposit_growth_yoy"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 50702


### 4.17. Crecimiento interanual del efectivo

<div style="text-align: justify;">

Se construye `cash_growth_yoy`, que mide la variación interanual del efectivo mantenido por cada institución.

Esta variable permite identificar cambios en la acumulación o reducción de recursos de disponibilidad inmediata. Una variación significativa del efectivo puede representar cambios en la estrategia de gestión de liquidez o una respuesta ante condiciones financieras particulares.

</div>

**Resultado:** Se genera `cash_growth_yoy` y se controlan valores nulos, infinitos y extremos.

In [38]:
# ============================================
# CRECIMIENTO INTERANUAL DEL EFECTIVO
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
).copy()

base_liquidez["cash_growth_yoy"] = (
    base_liquidez
    .groupby("id_rssd")["cash"]
    .pct_change(periods=4)
)

print("============================================")
print("CRECIMIENTO INTERANUAL DEL EFECTIVO")
print("============================================")

print(
    base_liquidez["cash_growth_yoy"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["cash_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["cash_growth_yoy"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["cash_growth_yoy"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO INTERANUAL DEL EFECTIVO
count    6.531290e+05
mean              inf
std               NaN
min     -5.268421e+01
25%     -1.842621e-01
50%      6.548043e-02
75%      4.381884e-01
max               inf
Name: cash_growth_yoy, dtype: float64

Valores infinitos: 167
Valores nulos: 45055

Percentiles:
0.010    -0.765489
0.050    -0.542088
0.500     0.065480
0.750     0.438188
0.900     1.149120
0.950     1.972946
0.990     5.690833
0.999    38.217829
Name: cash_growth_yoy, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\3374501279.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=4)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [39]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["cash_growth_yoy"] = (
    base_liquidez["cash_growth_yoy"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["cash_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["cash_growth_yoy"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 45222


### 4.18. Crecimiento interanual de valores negociables

<div style="text-align: justify;">

Se construye `securities_growth_yoy`, que representa la variación interanual de los valores negociables mantenidos por cada institución.

La variable permite capturar cambios en la composición de los activos líquidos y observar si las instituciones están incrementando o reduciendo su posición en instrumentos negociables.

Esta información complementa el crecimiento del efectivo y permite representar diferentes componentes de la posición de liquidez.

</div>

**Resultado:** Se genera `securities_growth_yoy` y se validan sus principales características estadísticas.

In [40]:
# ============================================
# CRECIMIENTO INTERANUAL DE VALORES
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
).copy()

base_liquidez["securities_growth_yoy"] = (
    base_liquidez
    .groupby("id_rssd")["securities"]
    .pct_change(periods=4)
)

print("============================================")
print("CRECIMIENTO INTERANUAL DE VALORES")
print("============================================")

print(
    base_liquidez["securities_growth_yoy"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["securities_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["securities_growth_yoy"].isna().sum()
)

print("\nPercentiles:")

print(
    base_liquidez["securities_growth_yoy"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO INTERANUAL DE VALORES
count    6.403430e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -1.065404e-01
50%      1.421315e-02
75%      1.881465e-01
max               inf
Name: securities_growth_yoy, dtype: float64

Valores infinitos: 3113
Valores nulos: 57841

Percentiles:
0.010   -0.750000
0.050   -0.359518
0.500    0.014213
0.750    0.188146
0.900    0.508041
0.950    0.929801
0.990    5.888964
0.999         NaN
Name: securities_growth_yoy, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\1081834877.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=4)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


In [41]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["securities_growth_yoy"] = (
    base_liquidez["securities_growth_yoy"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["securities_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["securities_growth_yoy"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 60954


### 4.19. Crecimiento interanual de pasivos

<div style="text-align: justify;">

Se construye `liabilities_growth_yoy`, que mide la evolución interanual de los pasivos totales.

Esta variable permite capturar cambios en el nivel de obligaciones de la institución y, por tanto, modificaciones en la estructura y magnitud de sus necesidades de financiamiento.

Su incorporación permite complementar la información de depósitos y analizar el comportamiento general de las fuentes de financiación del balance.

</div>

**Resultado:** Se genera `liabilities_growth_yoy` y se revisan sus valores nulos, infinitos y distribución.

In [42]:
# ============================================
# CRECIMIENTO INTERANUAL DE PASIVOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
).copy()

base_liquidez["liabilities_growth_yoy"] = (
    base_liquidez
    .groupby("id_rssd")["liab_tot"]
    .pct_change(periods=4)
)

print("============================================")
print("CRECIMIENTO INTERANUAL DE PASIVOS")
print("============================================")

print(
    base_liquidez["liabilities_growth_yoy"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["liabilities_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["liabilities_growth_yoy"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["liabilities_growth_yoy"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO INTERANUAL DE PASIVOS
count    6.535730e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -4.251873e-04
50%      4.957440e-02
75%      1.185702e-01
max               inf
Name: liabilities_growth_yoy, dtype: float64

Valores infinitos: 58
Valores nulos: 44611

Percentiles:
0.010    -0.204972
0.050    -0.083104
0.500     0.049574
0.750     0.118570
0.900     0.236189
0.950     0.387746
0.990     1.346993
0.999    12.624117
Name: liabilities_growth_yoy, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\669676615.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=4)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [43]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["liabilities_growth_yoy"] = (
    base_liquidez["liabilities_growth_yoy"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["liabilities_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["liabilities_growth_yoy"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 44669


### 4.20. Crecimiento interanual de activos

<div style="text-align: justify;">

Se construye `assets_growth_yoy`, que representa la variación interanual de los activos totales.

Esta variable permite capturar el crecimiento o contracción del tamaño del balance de cada institución. La dinámica de los activos puede estar relacionada con cambios en la actividad crediticia, la posición de liquidez y las necesidades de financiamiento.

Al utilizar una tasa de crecimiento se obtiene una medida comparable entre instituciones de diferente tamaño.

</div>

**Resultado:** Se genera `assets_growth_yoy` y se valida su distribución estadística.

In [44]:
# ============================================
# CRECIMIENTO INTERANUAL DE ACTIVOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
).copy()

base_liquidez["assets_growth_yoy"] = (
    base_liquidez
    .groupby("id_rssd")["assets"]
    .pct_change(periods=4)
)

print("============================================")
print("CRECIMIENTO INTERANUAL DE ACTIVOS")
print("============================================")

print(
    base_liquidez["assets_growth_yoy"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["assets_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["assets_growth_yoy"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["assets_growth_yoy"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO INTERANUAL DE ACTIVOS
count    653646.000000
mean          0.118934
std           5.659633
min          -1.000000
25%           0.003693
50%           0.049464
75%           0.113165
max        2658.088571
Name: assets_growth_yoy, dtype: float64

Valores infinitos: 0
Valores nulos: 44538

Percentiles:
0.010   -0.173229
0.050   -0.071389
0.500    0.049464
0.750    0.113165
0.900    0.222177
0.950    0.356923
0.990    1.044526
0.999    3.849218
Name: assets_growth_yoy, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\2085665179.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=4)


### 4.21. Crecimiento interanual de activos líquidos

<div style="text-align: justify;">

En esta etapa se construye primero la variable `liquid_assets`, definida como la suma del efectivo y los valores negociables. Posteriormente se calcula su crecimiento interanual.

Esta variable constituye una de las características centrales del proyecto, debido a que permite identificar cambios en la disponibilidad de recursos líquidos de cada institución a lo largo del tiempo.

El crecimiento interanual permite determinar si una institución está acumulando o reduciendo sus activos líquidos respecto al mismo período del año anterior.

</div>

**Resultado:** Se generan `liquid_assets` y `liquid_assets_growth_yoy`, incorporando información sobre la evolución histórica de la posición líquida.

In [45]:
# ============================================
# CRECIMIENTO INTERANUAL DE ACTIVOS LÍQUIDOS
# ============================================

# Activos líquidos
base_liquidez["liquid_assets"] = (
    base_liquidez["cash"] +
    base_liquidez["securities"]
)

# Crecimiento interanual
base_liquidez["liquid_assets_growth_yoy"] = (
    base_liquidez
    .groupby("id_rssd")["liquid_assets"]
    .pct_change(periods=4)
)

print("============================================")
print("CRECIMIENTO INTERANUAL DE ACTIVOS LÍQUIDOS")
print("============================================")

print(
    base_liquidez["liquid_assets_growth_yoy"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["liquid_assets_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["liquid_assets_growth_yoy"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["liquid_assets_growth_yoy"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO INTERANUAL DE ACTIVOS LÍQUIDOS
count    6.533880e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -8.233496e-02
50%      3.843619e-02
75%      2.060732e-01
max               inf
Name: liquid_assets_growth_yoy, dtype: float64

Valores infinitos: 64
Valores nulos: 44796

Percentiles:
0.010    -0.510223
0.050    -0.288047
0.500     0.038436
0.750     0.206073
0.900     0.486995
0.950     0.800143
0.990     2.405835
0.999    19.795033
Name: liquid_assets_growth_yoy, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\4284209978.py:15: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change(periods=4)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [46]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["liquid_assets_growth_yoy"] = (
    base_liquidez["liquid_assets_growth_yoy"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["liquid_assets_growth_yoy"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["liquid_assets_growth_yoy"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 44860


### 4.22. Ratio de activos líquidos sobre activos

<div style="text-align: justify;">

Se construye `liquid_assets_assets`, definido como la proporción de activos líquidos respecto al total de activos.

Este indicador representa la importancia relativa de los recursos líquidos dentro de la estructura total del balance. Constituye una medida fundamental para caracterizar la posición de liquidez de cada institución y posteriormente será utilizada como referencia para definir el deterioro de liquidez.

Una reducción de esta proporción puede indicar que los activos líquidos están creciendo a un ritmo inferior al crecimiento del balance o que están siendo utilizados para atender otras necesidades financieras.

</div>

**Resultado:** Se genera `liquid_assets_assets`, se controlan valores infinitos y nulos y se analiza su distribución.

In [47]:
# ============================================
# ACTIVOS LÍQUIDOS SOBRE ACTIVOS
# ============================================

base_liquidez["liquid_assets_assets"] = (
    base_liquidez["liquid_assets"] /
    base_liquidez["assets"]
)

print("============================================")
print("ACTIVOS LÍQUIDOS SOBRE ACTIVOS")
print("============================================")

print(
    base_liquidez["liquid_assets_assets"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["liquid_assets_assets"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["liquid_assets_assets"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["liquid_assets_assets"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

ACTIVOS LÍQUIDOS SOBRE ACTIVOS
count    6.981790e+05
mean              inf
std               NaN
min      0.000000e+00
25%      1.777675e-01
50%      2.718794e-01
75%      3.945811e-01
max               inf
Name: liquid_assets_assets, dtype: float64

Valores infinitos: 1
Valores nulos: 5

Percentiles:
0.010    0.030838
0.050    0.081001
0.500    0.271879
0.750    0.394581
0.900    0.529511
0.950    0.621388
0.990    0.843042
0.999    0.989732
Name: liquid_assets_assets, dtype: float64


d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [48]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["liquid_assets_assets"] = (
    base_liquidez["liquid_assets_assets"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["liquid_assets_assets"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["liquid_assets_assets"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 6


### 4.23. Depósitos a plazo sobre activos

<div style="text-align: justify;">

Se construye `time_deposits_assets`, que representa la proporción de depósitos a plazo respecto al total de activos.

Esta variable combina información sobre la estructura del fondeo y el tamaño del balance. Permite identificar qué importancia tienen los depósitos a plazo dentro de la estructura financiera total de cada institución.

</div>

**Resultado:** Se genera `time_deposits_assets` y se revisan sus principales características estadísticas.

In [49]:
# ============================================
# DEPÓSITOS A PLAZO SOBRE ACTIVOS
# ============================================

base_liquidez["time_deposits_assets"] = (
    base_liquidez["time_deposits"] /
    base_liquidez["assets"]
)

print("============================================")
print("DEPÓSITOS A PLAZO SOBRE ACTIVOS")
print("============================================")

print(
    base_liquidez["time_deposits_assets"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["time_deposits_assets"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["time_deposits_assets"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["time_deposits_assets"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

DEPÓSITOS A PLAZO SOBRE ACTIVOS
count    698178.000000
mean          0.326778
std           0.153121
min           0.000000
25%           0.217193
50%           0.328262
75%           0.434001
max           1.071576
Name: time_deposits_assets, dtype: float64

Valores infinitos: 0
Valores nulos: 6

Percentiles:
0.010    0.000000
0.050    0.073610
0.500    0.328262
0.750    0.434001
0.900    0.521786
0.950    0.575388
0.990    0.687003
0.999    0.827379
Name: time_deposits_assets, dtype: float64


### 4.24. Depósitos a la vista sobre activos

<div style="text-align: justify;">

Se construye `demand_deposits_assets`, que representa la participación de los depósitos a la vista dentro del total de activos.

Esta variable permite analizar conjuntamente la estructura de depósitos y el tamaño del balance, aportando información sobre la composición del fondeo de corto plazo en relación con los activos de la institución.

</div>

**Resultado:** Se genera `demand_deposits_assets` y se verifica su distribución y consistencia.

In [50]:
# ============================================
# DEPÓSITOS A LA VISTA SOBRE ACTIVOS
# ============================================

base_liquidez["demand_deposits_assets"] = (
    base_liquidez["demand_deposits"] /
    base_liquidez["assets"]
)

print("============================================")
print("DEPÓSITOS A LA VISTA SOBRE ACTIVOS")
print("============================================")

print(
    base_liquidez["demand_deposits_assets"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["demand_deposits_assets"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["demand_deposits_assets"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["demand_deposits_assets"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

DEPÓSITOS A LA VISTA SOBRE ACTIVOS
count    698178.000000
mean          0.139737
std           0.098968
min           0.000000
25%           0.071537
50%           0.121044
75%           0.185646
max           1.000000
Name: demand_deposits_assets, dtype: float64

Valores infinitos: 0


Valores nulos: 6

Percentiles:
0.010    0.000000
0.050    0.017451
0.500    0.121044
0.750    0.185646
0.900    0.264924
0.950    0.324765
0.990    0.467701
0.999    0.743000
Name: demand_deposits_assets, dtype: float64


### 4.25. Créditos sobre activos líquidos

<div style="text-align: justify;">

Se construye `loans_liquid_assets`, definido como la relación entre la cartera de créditos y los activos líquidos.

Este indicador permite aproximar la relación entre el volumen de activos crediticios y los recursos líquidos disponibles. Una relación elevada puede indicar una mayor magnitud de cartera respecto a la posición líquida de la institución.

La variable complementa los indicadores tradicionales de créditos sobre depósitos y créditos sobre activos, incorporando directamente la relación entre la actividad crediticia y los recursos líquidos.

</div>

**Resultado:** Se genera `loans_liquid_assets` y se controlan valores nulos, infinitos y extremos.

In [51]:
# ============================================
# CRÉDITOS SOBRE ACTIVOS LÍQUIDOS
# ============================================

base_liquidez["loans_liquid_assets"] = (
    base_liquidez["ln_tot"] /
    base_liquidez["liquid_assets"]
)

print("============================================")
print("CRÉDITOS SOBRE ACTIVOS LÍQUIDOS")
print("============================================")

print(
    base_liquidez["loans_liquid_assets"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["loans_liquid_assets"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["loans_liquid_assets"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["loans_liquid_assets"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRÉDITOS SOBRE ACTIVOS LÍQUIDOS
count    6.978600e+05
mean              inf
std               NaN
min      0.000000e+00
25%      1.349917e+00
50%      2.403170e+00
75%      4.183869e+00
max               inf
Name: loans_liquid_assets, dtype: float64

Valores infinitos: 39
Valores nulos: 324

Percentiles:
0.010      0.000000
0.050      0.470064
0.500      2.403170
0.750      4.183869
0.900      7.033402
0.950     10.031843
0.990     25.508144
0.999    141.591947
Name: loans_liquid_assets, dtype: float64


d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [52]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["loans_liquid_assets"] = (
    base_liquidez["loans_liquid_assets"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["loans_liquid_assets"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["loans_liquid_assets"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 363


### 4.26. Crecimiento trimestral de depósitos (QoQ)

<div style="text-align: justify;">

Se construye `deposit_growth_qoq`, que mide la variación porcentual de los depósitos respecto al trimestre inmediatamente anterior.

A diferencia del crecimiento interanual, esta variable permite capturar cambios de corto plazo en las fuentes de fondeo. Por ello, resulta particularmente relevante para un modelo orientado a anticipar episodios de deterioro de liquidez.

El cálculo se realiza individualmente para cada banco y respetando el orden cronológico de las observaciones.

</div>

**Resultado:** Se genera `deposit_growth_qoq`, permitiendo incorporar al modelo información sobre cambios recientes en los depósitos.

In [53]:
# ============================================
# CRECIMIENTO TRIMESTRAL DE DEPÓSITOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["deposit_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["deposits"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DE DEPÓSITOS")
print("============================================")

print(
    base_liquidez["deposit_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["deposit_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["deposit_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["deposit_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DE DEPÓSITOS
count    6.802290e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -1.322915e-02
50%      1.092896e-02
75%      4.077147e-02
max               inf
Name: deposit_growth_qoq, dtype: float64

Valores infinitos: 61
Valores nulos: 17955

Percentiles:
0.010   -0.116030
0.050   -0.057406
0.500    0.010929
0.750    0.040771
0.900    0.083267
0.950    0.128359
0.990    0.396607
0.999    3.016158
Name: deposit_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\516053646.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [54]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["deposit_growth_qoq"] = (
    base_liquidez["deposit_growth_qoq"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["deposit_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["deposit_growth_qoq"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 18016


### 4.27. Crecimiento trimestral del efectivo

<div style="text-align: justify;">

Se construye `cash_growth_qoq`, que mide la variación del efectivo respecto al trimestre anterior.

Esta variable permite identificar cambios recientes en los recursos de disponibilidad inmediata. Una reducción significativa del efectivo puede constituir una señal de presión sobre la posición de liquidez, mientras que un incremento puede reflejar una acumulación preventiva o cambios en la estructura del balance.

</div>

**Resultado:** Se genera `cash_growth_qoq` y se controlan valores nulos, infinitos y extremos.

In [55]:
# ============================================
# CRECIMIENTO TRIMESTRAL DEL EFECTIVO
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["cash_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["cash"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DEL EFECTIVO")
print("============================================")

print(
    base_liquidez["cash_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["cash_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["cash_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["cash_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DEL EFECTIVO
count    6.862210e+05
mean              inf
std               NaN
min     -3.986842e+01
25%     -1.679842e-01
50%      1.381356e-02
75%      2.459816e-01
max               inf
Name: cash_growth_qoq, dtype: float64

Valores infinitos: 111
Valores nulos: 11963

Percentiles:
0.010    -0.694572
0.050    -0.476927
0.500     0.013814
0.750     0.245982
0.900     0.648370
0.950     1.084320
0.990     2.890775
0.999    13.279801
Name: cash_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\3916213147.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [56]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["cash_growth_qoq"] = (
    base_liquidez["cash_growth_qoq"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["cash_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["cash_growth_qoq"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 12074


### 4.28. Crecimiento trimestral de activos líquidos

<div style="text-align: justify;">

Se construye `liquid_assets_growth_qoq`, que representa la variación trimestral de los activos líquidos.

Esta variable es especialmente relevante para el proyecto porque permite capturar cambios recientes en la posición líquida de las instituciones. Mientras que el nivel de activos líquidos muestra la situación en un momento determinado, su crecimiento permite identificar la dirección y velocidad del cambio.

La utilización conjunta de niveles y variaciones permite al modelo capturar tanto la posición actual como la dinámica reciente de liquidez.

</div>

**Resultado:** Se genera `liquid_assets_growth_qoq` y se validan sus valores nulos, infinitos y extremos.

In [57]:
# ============================================
# CRECIMIENTO TRIMESTRAL DE ACTIVOS LÍQUIDOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["liquid_assets_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["liquid_assets"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DE ACTIVOS LÍQUIDOS")
print("============================================")

print(
    base_liquidez["liquid_assets_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["liquid_assets_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["liquid_assets_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["liquid_assets_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DE ACTIVOS LÍQUIDOS
count    6.865360e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -5.482092e-02
50%      4.549199e-03
75%      7.952020e-02
max               inf
Name: liquid_assets_growth_qoq, dtype: float64

Valores infinitos: 39
Valores nulos: 11648

Percentiles:
0.010   -0.385688
0.050   -0.195254
0.500    0.004549
0.750    0.079520
0.900    0.204062
0.950    0.338112
0.990    0.962984
0.999    5.775487
Name: liquid_assets_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\3567541677.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [58]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["liquid_assets_growth_qoq"] = (
    base_liquidez["liquid_assets_growth_qoq"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["liquid_assets_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["liquid_assets_growth_qoq"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 11687


### 4.29. Crecimiento trimestral de créditos

<div style="text-align: justify;">

Se construye `loan_growth_qoq`, que mide la variación trimestral de la cartera crediticia.

Esta variable permite identificar cambios recientes en la actividad de colocación de créditos. Una expansión acelerada de la cartera puede generar necesidades adicionales de financiamiento y modificar la estructura de liquidez de una institución.

Por el contrario, una contracción puede reflejar cambios en la demanda de crédito, restricciones de financiamiento o modificaciones en la estrategia de gestión del balance.

</div>

**Resultado:** Se genera `loan_growth_qoq` y se realiza el correspondiente control de calidad.

In [59]:
# ============================================
# CRECIMIENTO TRIMESTRAL DE CRÉDITOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["loan_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["ln_tot"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DE CRÉDITOS")
print("============================================")

print(
    base_liquidez["loan_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["loan_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["loan_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["loan_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DE CRÉDITOS
count    6.762700e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -1.024949e-02
50%      1.317138e-02
75%      4.018228e-02
max               inf
Name: loan_growth_qoq, dtype: float64

Valores infinitos: 225
Valores nulos: 21914

Percentiles:
0.010   -0.119789
0.050   -0.056465
0.500    0.013171
0.750    0.040182
0.900    0.078560
0.950    0.120001
0.990    0.394317
0.999    4.895919
Name: loan_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\3615701576.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [60]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["loan_growth_qoq"] = (
    base_liquidez["loan_growth_qoq"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["loan_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["loan_growth_qoq"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 22139


### 4.30. Crecimiento trimestral de pasivos

<div style="text-align: justify;">

Se construye `liabilities_growth_qoq`, que representa la variación trimestral de los pasivos totales.

Esta variable permite identificar cambios recientes en las obligaciones de cada institución y complementar la información proporcionada por el crecimiento de depósitos.

Su inclusión permite representar posibles modificaciones de corto plazo en las necesidades de financiamiento y en la estructura del balance.

</div>

**Resultado:** Se genera `liabilities_growth_qoq` y se validan sus valores nulos e infinitos.

In [61]:
# ============================================
# CRECIMIENTO TRIMESTRAL DE PASIVOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["liabilities_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["liab_tot"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DE PASIVOS")
print("============================================")

print(
    base_liquidez["liabilities_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["liabilities_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["liabilities_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["liabilities_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DE PASIVOS
count    6.867480e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -1.193512e-02
50%      1.149285e-02
75%      3.967535e-02
max               inf
Name: liabilities_growth_qoq, dtype: float64

Valores infinitos: 54
Valores nulos: 11436

Percentiles:
0.010   -0.130368
0.050   -0.056322
0.500    0.011493
0.750    0.039675
0.900    0.081376
0.950    0.128590
0.990    0.436050
0.999    3.268740
Name: liabilities_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\151212567.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [62]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["liabilities_growth_qoq"] = (
    base_liquidez["liabilities_growth_qoq"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["liabilities_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["liabilities_growth_qoq"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 11490


### 4.31. Crecimiento trimestral de activos

<div style="text-align: justify;">

Se construye `assets_growth_qoq`, que mide la variación trimestral de los activos totales.

Esta variable permite identificar cambios recientes en el tamaño del balance de cada institución. Su utilización complementa las tasas de crecimiento de cartera, depósitos y pasivos, permitiendo analizar si la expansión del balance está siendo acompañada por cambios similares en sus principales fuentes y usos de fondos.

</div>

**Resultado:** Se genera `assets_growth_qoq` y se valida su distribución y consistencia.

In [63]:
# ============================================
# CRECIMIENTO TRIMESTRAL DE ACTIVOS
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["assets_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["assets"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DE ACTIVOS")
print("============================================")

print(
    base_liquidez["assets_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["assets_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["assets_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["assets_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DE ACTIVOS
count    686845.000000
mean          0.027279
std           1.568401
min          -1.000000
25%          -0.009406
50%           0.011594
75%           0.036879
max         961.909369
Name: assets_growth_qoq, dtype: float64

Valores infinitos: 0
Valores nulos: 11339

Percentiles:
0.010   -0.101704
0.050   -0.048710
0.500    0.011594
0.750    0.036879
0.900    0.073350
0.950    0.112488
0.990    0.309005
0.999    1.189483
Name: assets_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\4234735260.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()


### 4.32. Crecimiento trimestral de valores negociables

<div style="text-align: justify;">

Se construye `securities_growth_qoq`, que mide la variación trimestral de los valores negociables.

Esta variable permite capturar cambios recientes en uno de los componentes principales de los activos líquidos. Una modificación importante en esta posición puede representar una estrategia de acumulación o utilización de activos negociables frente a cambios en las condiciones financieras.

</div>

**Resultado:** Se genera `securities_growth_qoq` y se realizan controles sobre valores nulos, infinitos y extremos.

In [64]:
# ============================================
# CRECIMIENTO TRIMESTRAL DE VALORES
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["securities_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["securities"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DE VALORES")
print("============================================")

print(
    base_liquidez["securities_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["securities_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["securities_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["securities_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DE VALORES
count    6.704770e+05
mean              inf
std               NaN
min     -1.000000e+00
25%     -4.585810e-02
50%     -3.115555e-03
75%      5.329519e-02
max               inf
Name: securities_growth_qoq, dtype: float64

Valores infinitos: 1274
Valores nulos: 27707

Percentiles:
0.010   -0.415162
0.050   -0.169789
0.500   -0.003116
0.750    0.053295
0.900    0.164492
0.950    0.293497
0.990    1.113617
0.999         NaN
Name: securities_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\1439130680.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4608: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


In [65]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["securities_growth_qoq"] = (
    base_liquidez["securities_growth_qoq"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["securities_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["securities_growth_qoq"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 28981


### 4.33. Crecimiento trimestral del patrimonio

<div style="text-align: justify;">

Finalmente, se construye `equity_growth_qoq`, que representa la variación trimestral del patrimonio de cada institución.

Esta variable permite incorporar información sobre la evolución reciente de la posición patrimonial. Los cambios en el patrimonio pueden reflejar resultados acumulados, aportes de capital u otros movimientos financieros que pueden modificar la capacidad de absorción de pérdidas y la estructura financiera de la institución.

Su incorporación complementa las variables de liquidez, fondeo, activos, pasivos y calidad de cartera utilizadas en la construcción del conjunto de predictores.

</div>

**Resultado:** Se genera `equity_growth_qoq` y se validan sus valores nulos, infinitos y distribución.

In [66]:
# ============================================
# CRECIMIENTO TRIMESTRAL DEL PATRIMONIO
# ============================================

base_liquidez = base_liquidez.sort_values(
    ["id_rssd", "date"]
)

base_liquidez["equity_growth_qoq"] = (
    base_liquidez.groupby("id_rssd")["equity"]
    .pct_change()
)

print("============================================")
print("CRECIMIENTO TRIMESTRAL DEL PATRIMONIO")
print("============================================")

print(
    base_liquidez["equity_growth_qoq"].describe()
)

print(
    "\nValores infinitos:",
    np.isinf(
        base_liquidez["equity_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos:",
    base_liquidez["equity_growth_qoq"]
    .isna()
    .sum()
)

print("\nPercentiles:")

print(
    base_liquidez["equity_growth_qoq"]
    .quantile(
        [
            0.01,
            0.05,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)

CRECIMIENTO TRIMESTRAL DEL PATRIMONIO
count    6.868450e+05
mean             -inf
std               NaN
min              -inf
25%     -2.073906e-03
50%      1.486729e-02
75%      3.214370e-02
max      1.025700e+04
Name: equity_growth_qoq, dtype: float64

Valores infinitos: 1
Valores nulos: 11339

Percentiles:
0.010   -0.187490
0.050   -0.060309
0.500    0.014867
0.750    0.032144
0.900    0.057030
0.950    0.088516
0.990    0.337875
0.999    1.896056
Name: equity_growth_qoq, dtype: float64


C:\Users\LENOVO YOGA\AppData\Local\Temp\ipykernel_33268\1987377598.py:11: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
d:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [67]:
# ============================================
# CORRECCIÓN DE VALORES INFINITOS
# ============================================

base_liquidez["equity_growth_qoq"] = (
    base_liquidez["equity_growth_qoq"]
    .replace([np.inf, -np.inf], np.nan)
)

print(
    "Valores infinitos después de corregir:",
    np.isinf(
        base_liquidez["equity_growth_qoq"]
    ).sum()
)

print(
    "Valores nulos después de corregir:",
    base_liquidez["equity_growth_qoq"]
    .isna()
    .sum()
)

Valores infinitos después de corregir: 0
Valores nulos después de corregir: 11340


## 5. Guardar base de indicadores

<div style="text-align: justify;">

Una vez finalizada la construcción de los indicadores financieros y variables de crecimiento, se prepara la base resultante para ser utilizada en las siguientes etapas del proyecto.

La información generada constituye una versión enriquecida de la base original, ya que incorpora variables relativas y dinámicas que permiten representar diferentes dimensiones del comportamiento financiero de las instituciones bancarias.

La base se mantiene dentro de la estructura del proyecto `LiquiditySense`, específicamente en la carpeta destinada a los datos procesados. Esta organización permite mantener la trazabilidad entre la información original, la base depurada y la base enriquecida mediante ingeniería de variables.

Es importante señalar que esta etapa corresponde a la construcción de predictores financieros. La definición formal de la variable objetivo y el tratamiento específico de las observaciones necesarias para realizar la predicción del siguiente período se completarán como parte de la construcción del problema supervisado.

</div>

**Resultado:** Se obtiene una base enriquecida con indicadores de liquidez, estructura financiera y tasas de crecimiento, preparada para las siguientes etapas del proyecto.

In [68]:
from pathlib import Path

# Carpeta raíz del proyecto
ruta_proyecto = Path(
    r"D:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense"
)

# Carpeta para bases procesadas
ruta_procesada = ruta_proyecto / "data" / "processed"

# Crear carpeta si no existe
ruta_procesada.mkdir(
    parents=True,
    exist_ok=True
)

print("Carpeta de salida:")
print(ruta_procesada)

Carpeta de salida:
D:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\data\processed


In [69]:
# ============================================
# GUARDAR BASE PROCESADA — PASO 01
# ============================================

ruta_salida = ruta_procesada / "base_liquidez_indicadores.csv"

base_liquidez.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print("============================================")
print("BASE GUARDADA CORRECTAMENTE")
print("============================================")

print(f"Ruta: {ruta_salida}")
print(f"Registros: {len(base_liquidez):,}")
print(f"Variables: {base_liquidez.shape[1]:,}")

BASE GUARDADA CORRECTAMENTE
Ruta: D:\Edwin\DIPLOMADO INGENIERIA DE DATOS\Machine Learning II\Proyecto\LiquiditySense\data\processed\base_liquidez_indicadores.csv
Registros: 698,184
Variables: 55


## 6. Conclusión

<div style="text-align: justify;">

La etapa de **ingeniería de variables** permitió transformar la información financiera original de las instituciones bancarias en un conjunto estructurado de variables capaces de representar diferentes dimensiones relacionadas con su comportamiento de liquidez. A partir de las cuentas financieras originales se construyeron indicadores de cobertura, estructura financiera, calidad de cartera, solvencia y evolución temporal, incorporando tanto relaciones entre cuentas del balance como variaciones interanuales (YoY) y trimestrales (QoQ).

Un elemento fundamental de esta etapa fue considerar la **naturaleza longitudinal de los datos**, realizando los cálculos dentro de cada institución bancaria y respetando el orden cronológico de las observaciones. Esto permitió obtener medidas dinámicas consistentes con la periodicidad trimestral de la información y constituye una condición necesaria para el posterior desarrollo de un modelo predictivo temporal.

Asimismo, durante la construcción de las variables se contemplaron situaciones propias de la información financiera, como denominadores iguales a cero, valores infinitos y períodos con información insuficiente para calcular determinadas tasas de crecimiento. Estos casos fueron tratados sin introducir valores artificiales, preservando la integridad de la información para su posterior procesamiento.

Las variables generadas permiten caracterizar no solamente el nivel de liquidez de cada institución, sino también su **dinámica y posibles señales de deterioro**. En particular, los indicadores relacionados con activos líquidos, depósitos, efectivo, valores y sus respectivas tasas de crecimiento proporcionan información relevante para analizar cambios en la posición de liquidez a través del tiempo.

Finalmente, la base resultante constituye un insumo fundamental para las siguientes etapas del proyecto **LiquiditySense**, ya que proporciona las variables explicativas necesarias para realizar el análisis exploratorio, definir formalmente el evento de deterioro de liquidez y posteriormente entrenar y evaluar los modelos de Machine Learning. La metodología aplicada mantiene como principio la separación temporal de la información y la prevención de *data leakage*, contribuyendo a que los resultados posteriores representen de manera más realista la capacidad del modelo para anticipar episodios de deterioro de liquidez.

</div>
